In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
## Data Ingestion
import pandas as pd

# path from output
review_path = '/kaggle/input/datasets/organizations/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json'

# Use chunking to load a manageable sample
# "large-scale data ingestion"
reader = pd.read_json(review_path, lines=True, chunksize=50000)
df = next(reader)

# Quick check: columns and the text being analyzed
print(f"Dataset loaded with {df.shape[0]} rows.")
print(df[['stars', 'text']].head())

In [ ]:
## Text Vectorization
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. Clean the data
X = df['text']
y = df['stars']

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Text has been successfully converted to numerical vectors!")

In [ ]:
## Model Evaluation
# 1. Initialize and Train the Model
model = LinearRegression()
model.fit(X_train_tfidf, y_train)

# 2. Make Predictions on the Test Set
predictions = model.predict(X_test_tfidf)

# 3. Calculate the Margin of Error (Mean Absolute Error)
mae = mean_absolute_error(y_test, predictions)

print(f"Model Training Complete!")
print(f"Mean Absolute Error: {mae:.4f} stars")

# 4. Quick Test: real prediction vs the actual rating
sample_index = 0
print(f"\nSample Review: {X_test.iloc[sample_index][:100]}...")
print(f"Actual Rating: {y_test.iloc[sample_index]} stars")
print(f"Predicted Rating: {predictions[sample_index]:.2f} stars")